# 不计分实验：多类别分类——一对多

当类别多于两个时，一对多（One vs All）是一种可供选择的方法。
![pic](./图表/onevsmany.png)

## 目标
在本实验中，你将：
- 运用你已经开发出的函数（compute_cost、compute_gradient、predict、gradient_descent）做二项（是/否）分类
- 编写一个多类别预测程序，在 n 个二项决策之间做选择
- 运用决策边界绘图的逻辑


# 目录
- [工具](#tools)
- [数据集](#dataset)
- [一对多实现](#ova)



# 多类别分类：一对多（OVA）
在本实验中，我们将探索当类别多于两个时如何使用一对多方法进行分类。该技术是我们之前使用的两类（二项）逻辑回归的扩展。

在二项逻辑回归中，我们训练一个模型来区分样本属于某个类别还是不属于。一对多（OVA）通过训练 $n$ 个模型来扩展这一方法。每个模型负责识别一个类别。给定类别的模型是这样训练的：把训练集重新表述为“识别一个类别为正类、其余全部为负类”。做预测时，一个样本会被所有 $n$ 个模型处理，选出预测输出最大的那个模型。

在本实验中，我们将构建一个 OVA 分类器。
## 工具
- 我们将利用之前的工作来构建和训练模型。这些程序已提供给你。
- 绘制决策边界和数据集很有帮助。生成这些图涉及不少细节，因此下面提供了辅助程序。
        - plot_mc_decision_boundary() 会使用你在本次作业中要编写的预测程序 `predict_mc`
- 我们将创建一个多类别数据集。这里用到常用的 [`SkLearn`](https://scikit-learn.org/stable/) 程序。

In [ ]:
from lab_utils import *
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
import copy
import math

这些程序已提供，但回顾它们的运作方式很有启发。绘图程序常常用到许多冷门但有用的 numpy 用法。绘制决策边界用到了 `matplotlib` 的等高线图。等高线图会在数值发生变化的分界处画线，这一能力被用来标出决策的变化。简单来说，该程序有三步：
- 在二维网格上创建细密的点位网格，并把这些点组成数组。
- 对每个点做预测。在本例中，这包括投票选出最佳预测。
- 用等高线图绘制网格与预测值（`Z`）的关系。

In [ ]:
#绘制多类别决策边界
def plot_mc_decision_boundary(X,nclasses, W, b , predict_mc_function, class_labels=None, legend=False):

    # 创建用于绘图的网格点
    h = 0.1  # 网格的步长
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    points = np.c_[xx.ravel(), yy.ravel()]

    #为网格中的每个点做预测
    Z = predict_mc_function(points,W,b)
    Z = Z.reshape(xx.shape)

    #等高线图突出显示数值之间的边界——本例中即类别之间的边界
    plt.figure()
    plt.contour(xx, yy, Z, colors='g') 
    plt.axis('tight')


In [ ]:
# 绘制多类别训练点
def plot_mc_data(X, y, class_labels=None, legend=False):
    classes = np.unique(y)
    for i in classes:
        label = class_labels[i] if class_labels else "class {}".format(i)
        idx = np.where(y == i)
        plt.scatter(X[idx, 0], X[idx, 1],  cmap=plt.cm.Paired,
                    edgecolor='black', s=20, label=label)
    if legend: plt.legend()

我们提供你在之前实验中开发出的程序，用于创建并拟合/训练模型。你可以随意把它们替换成自己的版本。（请保留一份原版以防万一。）

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, predict_function, alpha, num_iters): 
    """执行批量梯度下降以学习 theta。以学习率 alpha 走
    num_iters 个梯度步来更新 theta
   
    参数：
      X :    (array_like 形状 (m,n)
      y :    (array_like 形状 (m,1))
      w_in : (array_like 形状 (n,1)) 模型参数的初始值
      b_in : (scalar)                 模型参数的初始值
      cost_function:                  用于计算代价的函数
      gradient_function:              用于计算梯度的函数
      predict_function                用于计算模型输出的函数
      alpha : (float)                 学习率
      num_iters : (int)               运行梯度下降的迭代次数
    返回
      w : (array_like 形状 (n,)) 运行梯度下降后更新得到的模型参数值
      b : (scalar)                运行梯度下降后更新得到的模型参数值"""
    
    # 训练样本数
    m = len(X)
    
    # 用于存放代价 J 以及每次迭代的 w，主要供后续绘图使用
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  # 避免在函数内修改全局变量 w
    b = b_in
    
    for i in range(num_iters):

        # 计算梯度并更新参数
        dJdb,dJdw = gradient_function(X, y, w, b, predict_function)   ##None

        # 用 w、b、alpha 和梯度更新参数
        w = w - alpha * dJdw               ##None
        b = b - alpha * dJdb               ##None

        # 每次迭代保存代价 J
        if i<100000:      # 防止资源耗尽
            cost =  cost_function(X, y, w, b)
            J_history.append(cost)

        # 每 10 次迭代打印一次代价；若迭代次数少于 10，则打印所有迭代
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
    #print(w,b,len(J_history), len(w_history) )
    return w, b, J_history, w_history #返回 w 以及用于绘图的 J、w 历史

In [ ]:
def compute_cost_logistic_matrix(X, y, w, b):
    """计算所有样本上的代价
    参数：
      X : (array_like 形状 (m,n)) 数据，m 个样本，n 个特征
      y : (array_like 形状 (m,1)) 目标值
      w : (array_like 形状 (n,1)) 模型参数取值
      b : (array_like 形状 (n,1)) 模型偏置参数取值
    返回：
      total_cost: (scalar)         代价"""
    m = X.shape[0]
    
    f = sigmoid(X @ w + b)
    total_cost = (1/m)*(np.dot(-y.T, np.log(f)) - np.dot((1-y).T, np.log(1-f)))
    
    return total_cost[0,0]

In [ ]:
def predict_logistic_matrix(X, w, b): 
    """使用线性回归做单样本预测
    参数：
      x : (array_like 形状 (m,n)) 特征值，房屋面积、卧室数……
      w : (array_like 形状 (n,)) 用于预测的参数
      b : (scalar               ) 用于预测的参数
    返回
      p: ((array_like 形状 (n,)  预测值"""
    p = sigmoid(X @ w + b )         
    
    return(p)    

In [ ]:
def predict_thresh(X, w, b, threshold=0.5): 
    """使用学习到的逻辑回归参数 w、b 预测标签是 0 还是 1。对 sigmoid 的输出取阈值来预测 1 或 0。
   
    参数
    ----------
    X : array_like
        形状 (m, n)
   
    w : array_like
        模型参数
        形状 (n, 1)
    b : scalar
   
    返回
    -------

    p: array_like
        形状 (m,)
        以 0.5 为阈值对 X 做出的预测"""
    # 训练样本数
    m = X.shape[0]   
    p = np.zeros(m)
   
    for i in range(m):
        f_w = sigmoid(np.dot(w.T, X[i])+b)
        p[i] = f_w >= threshold
    
    return p

In [ ]:
def compute_gradient_logistic_matrix(X, y, w, b, predict_function): 
    """计算线性回归的梯度
 
    参数：
      X : (array_like 形状 (m,n)) 变量，例如房屋面积
      y : (array_like 形状 (m,1)) 实际值
      w : (array_like 形状 (n,1)) 模型参数取值
      b : (scalar )                模型参数取值
      predict_function: (function) 用于做预测的函数
    返回
      dJdw: (array_like 形状 (n,1)) 代价对参数 w 的梯度。
      dJdb: (scalar)                代价对参数 b 的梯度。"""
    m,n = X.shape
    f_wb = predict_function(X, w, b) 
    err  = f_wb - y                 
    dJdw = (1/m) * (X.T @ err)     
    dJdb = (1/m) * np.sum(err)     
        
    return dJdb,dJdw

<a name='dataset'></a>
##  数据集
下面我们用 `SkLearn` 的一个工具创建 3 个数据“团块”（blob）。这是快速便捷地创建分类数据的办法。利用 NumPy 的 [`np.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)，我们可以查看类别的数量和取值。
- **注意**，我们创建的是 3 个类别

In [ ]:
# 为分类创建三类别数据集
centers = [[-5, 0], [0, 4.5], [5, -1]]
X_train, y_train = make_blobs(n_samples=500, centers=centers, cluster_std=0.85,random_state=40)


In [ ]:
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

In [ ]:
# 显示数据集中的类别
print(f"unique classes {np.unique(y_train)}")
# 展示类别的表示方式
print(f"unique classes {y_train[:10]}")
# 显示数据集的形状
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

在创建一对多训练集时，对每个类别，你需要从 `y_train` 创建一个“二元”训练集，即把该类别中的所有样本值设为 `1`。NumPy 有一个特性可以让这件事很方便，如下所示：

In [ ]:
y_cat_2 = (y_train == 2) + 0
print(y_cat_2[:10])

你需要二值（0、1）取值。

In [ ]:
y_cat_2 = (y_train == 2).astype(float)
print(y_cat_2[:10])

<a name='ova'></a>
##  一对多实现

你将分三步实现 OVA 算法。
- 创建并训练三个“模型”。每个模型都被训练用来选出三个类别中的一个。
- 编写一个程序 `predict_mc`，用它来做出预测并选出最佳答案。
- 用该预测程序绘制决策边界。

### 第 1 步：创建并训练 3 个模型。
涉及的步骤在以往使用梯度下降的实验中你已经熟悉。
对每个类别：
- 分离出与当前类别对应的目标 `y`。
- 创建 `w_init` 和 `b_init`，即参数的初始值。
- 调用梯度下降。alpha=1e-2、num_iters=1000 效果不错。
    - 对 gradient_descent 的调用有若干参数，该程序就在上面。用*提示*中的代码复核你的解答。注意梯度下降是如何运用你迄今开发出的所有程序的。
    - 你将调用 `predict_logistic_matrix` 来做预测，它不包含阈值逻辑。你在不带阈值的情况下训练，之后使用模型时再加上阈值。
    - 它返回参数 $w$ 和 $b$。$w$ 和 $b$ 构成你的*模型*，你会把它们存到数组里。重要的是，这个数组*每个模型占一列*。这种安排便于使用矩阵运算，你在后续实验中会熟悉它。
- 用训练数据和你的模型（$w$、$b$）调用 predict 来绘制训练结果。

下面是一个遍历各类别的 for 循环。
- 创建一个目标数组，把当前类别设为一、其他都设为零。
- 你的代码
- 绘制对数据的这种解释
- 绘制预测值
把 `None` 替换成你的代码。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>


```python
classes=np.unique(y_train)   # three classes, [0,1,2]
m,n = X_train.shape          # number of examples, number of features
c = len(classes)             # number of classe

# storage for our models (w), one column per class
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)
plt.figure(figsize=(14, 14))             

for i in classes:
    yc = (y_train == classes[i]).astype(float)
    yc = yc.reshape(-1,1)  

    ### START CODE HERE ### 
    w_init = np.zeros((2,1))                                                               
    b_init = 0.                                                                            
    w_final, b_final,_,_ = gradient_descent(X_train, yc, w_init, b_init,                   
                                      compute_cost_logistic_matrix,                       
                                      compute_gradient_logistic_matrix,                  
                                      predict_logistic_matrix,                            
                                      alpha = 1e-2, num_iters=1000)                         
    W_models[:,i] = w_final[:,0]                                                          
    b_models[i] = b_final                                                                 
    pred =  predict_thresh(X_train, w_final,b_final )                                    

    ### END CODE HERE ###         

    #Left Side, training data in All vs i
    ax = plt.subplot(3,2, 2*i + 1)
    plot_mc_data(X_train, yc,legend=True); plt.title(f"Training Classes, class {i}"); 

    #Right Side, model i's prediction after training
    ax = plt.subplot(3,2, 2*i + 2)
    plot_mc_data(X_train,pred,legend=True); plt.title("Predicted Classes after training");
plt.show
```

</details>

In [ ]:
classes=np.unique(y_train)   # 三个类别，[0,1,2]
m,n = X_train.shape          # 样本数，特征数
c = len(classes)             # 类别数

# 存放我们的模型（w），每个类别一列
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)
plt.figure(figsize=(14, 14))             

for i in classes:
    yc = (y_train == classes[i]).astype(float)
    yc = yc.reshape(-1,1)  

    ### 在此处编写代码 ###

    w_init = None  
    b_init = None  
    # 调用梯度下降，用提示核对你的解答
    w_final, b_final,_,_ = None  
    ### 代码编写结束 ###
    W_models[:,i] = w_final[:,0]                                                           
    b_models[i] = b_final                                                                  
    pred =  predict_thresh(X_train, w_final,b_final )                                      

    #左侧，把训练数据放入“全部 vs 第 i 个”的对比
    ax = plt.subplot(3,2, 2*i + 1)
    plot_mc_data(X_train, yc,legend=True); plt.title(f"Training Classes, class {i}"); 

    #右侧，第 i 个模型训练后的预测
    ax = plt.subplot(3,2, 2*i + 2)
    plot_mc_data(X_train,pred,legend=True); plt.title("Predicted Classes after training");
plt.show()

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

 ![asdf](./图表/C1W3_trainvpredict.PNG)

既然我们已经训练好 3 个模型，接下来编写一个程序来选出最佳预测。回忆一下，该操作包括
- 对每个模型做一次预测
- 选出最大的预测值

-第 1 步：给定 $X$ 以及矩阵 `W_model` 和 `b_models`，执行一次预测，得到三个预测值。这可以用向量化形式实现，如下图所示。这不是一个平凡的操作，值得花时间理解它在做什么。也可以用 for 循环实现。
![pic](./图表/C1W3_mcpredict.PNG)  
-第 2 步：用 `np.argmax(axis=1)` 返回取值最高的预测所对应的**类别**。注意类别是 [0,1,2] 之一，而 `np.argmax` 返回的索引恰好也是 [0,1,2]。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>


```python
def predict_mc(X,W,b, verbose = False):
    """
    Computes n predictions and selects the best.
    Args:
      X : (array_like Shape (m,n)) feature values used in prediction.  
      W : (array_like Shape (n,c)) Matrix of parameter. Each column represents 1  model
      b : (array_like Shape (c, )) vector of bias parameter. Each column represents 1  model
    Returns
      sclass: (array_like Shape (m,1)) The selected class the values belong in. Values 0 to c.
    """
    ### START CODE HERE ### 
    ### BEGIN SOLUTION ###  
    z_wb = X @ W + b               #Matrix multiply and add  ##None
    f_wb = sigmoid(z_wb)              #sigmoid                  ##None
    pclass = f_wb.argmax(axis=1)      #argmax                   ##None
    ### END SOLUTION ###  
    ### END CODE HERE ### 
    if verbose: print("z_wb.shape",z_wb.shape); print(z_wb)
    if verbose: print("pclass",pclass)
    return(pclass)
```

</details>

In [ ]:
def predict_mc(X,W,b, verbose = False):
    """计算 n 个预测并选出最好的一个。
    参数：
      X : (array_like 形状 (m,n)) 用于预测的特征值。
      W : (array_like 形状 (n,c)) 参数矩阵。每一列代表 1 个模型
      b : (array_like 形状 (c, )) 偏置参数向量。每一列代表 1 个模型
    返回
      sclass: (array_like 形状 (m,1)) 这些取值所属的被选中的类别。取值从 0 到 c。"""
    ### 在此处编写代码 ###

    #矩阵乘法并相加
    #sigmoid
    #argmax
    ### 代码编写结束 ###
    if verbose: print("z_wb.shape",z_wb.shape); print(z_wb)
    if verbose: print("pclass",pclass)
    return(pclass)

In [ ]:
#测试你的模型
tmp_X = np.array([[-2.,-6.],[6,0],[-2,6]])                            #(2,2)
tmp_w = np.array([[-1.117, 0.103, 0.963], [-0.863, 1.155, -0.954]])   #(2,3)
tmp_b = np.array([-0.267 -1.4577 -0.238])                             #(3, )
print(tmp_X.shape, tmp_w.shape, tmp_b.shape)

tmp_fw = predict_mc(tmp_X,tmp_w,tmp_b, verbose = True)
print(tmp_fw)

<details>
<summary>
    <b>**预期输出**：</b>
</summary>


```
(3, 2) (2, 3) (1,)
z_wb.shape (3, 3)
[[ 5.4493 -9.0987  1.8353]
 [-8.6647 -1.3447  3.8153]
 [-4.9067  4.7613 -9.6127]]
pclass [0 2 1]
[0 2 1]
```


既然我们能为任意点做出预测，现在就可以绘制出显示决策边界的图了。`plot_mc_decision_boundary` 依靠你的 `predict_mc` 来工作。

In [ ]:
#绘制决策边界。传入我们的模型——与每个模型关联的 w 和 b，以及 predict_mc
plot_mc_decision_boundary(X_train,3, W_models, b_models, predict_mc)
plt.title("model decision boundary vs original training data")

#把原始数据加到决策边界上
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

<details>
<summary>
    <b>**预期输出**：</b>
</summary>

![sdf](./图表/C1W3_boundary.PNG)

成功了！你现在已经构建了一个多类别分类器。

让我们试试另一种情况。我们把团块稍微挪一挪：
## 第二个测试用例

In [ ]:
# 为分类创建三类别数据集
centers = [[-5, 0], [0, 1], [5, -1]]
X_train, y_train = make_blobs(n_samples=500, centers=centers, cluster_std=1.2,random_state=40)


In [ ]:
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()

In [ ]:
# 显示数据集中的类别
print(f"unique classes {np.unique(y_train)}")
# 显示数据集的形状
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

观察上面的图，你能看出我们当前的做法有什么潜在问题吗？

把上面各部分拼起来，或者编写子程序，做出像第一个例子那样的决策边界图。

<details>
  <summary><font size="2" color="darkgreen"><b>提示</b></font></summary>


```python
classes=np.unique(y_train)   # three classes, [0,1,2]
m,n = X_train.shape          # number of examples, number of features
c = len(classes)             # number of classe

# storage for our models (w), one column per class
W_models = np.zeros((n,len(classes)))   
b_models = np.zeros(c)

for i in classes:
    yc = (y_train==classes[i]) + 0
    yc = yc.reshape(-1,1)

    w_init = np.zeros((2,1))   
    b_init = 0.
    w_final, b_final,_,_ = gradient_descent(X_train, yc, w_init, b_init,
                                      compute_cost_logistic_matrix, 
                                      compute_gradient_logistic_matrix, 
                                      predict_logistic_matrix,
                                      alpha = 1e-2, num_iters=1000)     
    W_models[:,i] = w_final[:,0]
    b_models[i] = b_final
    pred =  predict_thresh(X_train, w_final,b_final ) 

plot_mc_decision_boundary(X_train,3, W_models, b_models, predict_mc)
plt.title("model decision boundary vs original training data")

#add the original data to the decison boundary
plot_mc_data(X_train,y_train,["blob one", "blob two", "blob three"], legend=True)
plt.show()
```

</details>

In [ ]:
#在此处重写代码



<details>
<summary>
    <b>**预期输出**：</b>
</summary>

![asdf](./图表/C1W3_example2.PNG)
    
我们将在下一个实验中研究带多项式特征的逻辑回归。那将使我们能够处理纯线性解不够用的情况。

本 notebook 参考了 scikit-learn.org 上的一个示例。作者是 Tom Dupre la Tour <tom.dupre-la-tour@m4x.org>